In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/credit_risk.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 5
TEST_SIZE = 0.25


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))
data.head()


In [ ]:
data.info()


In [ ]:
# Распределение late_payments
plt.figure(figsize=(8, 4))
sns.histplot(data=data, x='late_payments', hue='default', bins=15, multiple='stack', palette='rocket')
plt.title('late_payments по default', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / 'credit_late_hist.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
data.describe().round(2)


In [ ]:
print("default rate:", round(data["default"].mean(), 3))


In [ ]:
# ======================
# 2. СЫРЫЕ ПРИЗНАКИ
# ======================
raw_cols = ["age", "income", "debt_ratio", "num_cards", "late_payments", "employment_years"]
X_raw = data[raw_cols].copy()
y = data["default"]
X_raw.head()


In [ ]:
# ======================
# 3. FEATURE ENGINEERING
# ======================
fe = data[raw_cols].copy()

# Отношения и взаимодействия
fe["debt_to_income"] = fe["debt_ratio"] * fe["income"] / 1000  # условный долг
fe["income_per_card"] = fe["income"] / (fe["num_cards"] + 1)
fe["late_per_year"] = fe["late_payments"] / (fe["employment_years"] + 1)
fe["stress"] = fe["debt_ratio"] * fe["late_payments"]

# Биннинг возраста
fe["age_bin"] = pd.cut(fe["age"], bins=[17, 30, 45, 60, 100], labels=False)

# One-hot для бинов
fe = pd.get_dummies(fe, columns=["age_bin"], drop_first=True)

print("FE columns:", list(fe.columns))
fe.head()


In [ ]:
# Корреляция новых признаков с default
corr_cols = ["debt_to_income", "income_per_card", "late_per_year", "stress", "default"]
tmp = fe.copy()
tmp["default"] = y
corr = tmp[corr_cols].corr()["default"].drop("default").sort_values()

plt.figure(figsize=(8, 4))
sns.barplot(x=corr.values, y=corr.index, palette="coolwarm", hue=corr.values, legend=False)
plt.title("Корреляция FE-признаков с default", fontsize=14, pad=15)
plt.xlabel("corr", fontsize=12)
plt.axvline(0, color="black", lw=0.8)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "credit_fe_corr.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 4. СРАВНЕНИЕ RAW vs FE
# ======================
def eval_auc(name, X):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000)),
    ])
    pipe.fit(Xtr, ytr)
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:, 1])
    print(f"{name:6s} AUC={auc:.3f}")
    return auc, pipe, Xte, yte

auc_raw, _, _, _ = eval_auc("raw", X_raw)
auc_fe, pipe_fe, Xte_fe, yte_fe = eval_auc("fe", fe)


In [ ]:
# Таблица сравнения
compare = pd.DataFrame({
    "variant": ["raw", "fe"],
    "auc": [auc_raw, auc_fe],
})
compare


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=compare, x="variant", y="auc", palette="viridis", hue="variant", legend=False)
plt.title("Влияние feature engineering на AUC", fontsize=14, pad=15)
plt.ylabel("ROC-AUC", fontsize=12)
plt.ylim(0.5, 1.0)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for i, row in compare.iterrows():
    plt.text(i, row["auc"] + 0.01, f"{row['auc']:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.savefig(FIG / "credit_fe_auc.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print(classification_report(yte_fe, pipe_fe.predict(Xte_fe), digits=3))
print("Отношения и бины часто помогают линейным моделям сильнее, чем деревьям.")
